# 🧠 TP53 Deep Learning Architecture & Predictive Modelling

Exploratory deep-learning framework for comparative TP53 sequence analysis.

**Researcher:** Ritika Rajendra Rawat  
**Degree:** MSc Bioinformatics  
**Institution:** University of Mumbai

> ⚠️ This is a methodological prototype. The current small sequence dataset is not sufficient for validated supervised prediction or clinical claims.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO
from sklearn.decomposition import PCA
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import pad_sequences

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

In [ ]:
def find_repository_root():
    current = Path.cwd().resolve()
    for path in [current, current.parent, current.parent.parent, current.parent.parent.parent]:
        if (path / "data").exists() and (path / "README.md").exists():
            return path
    raise FileNotFoundError("Repository root not found")

ROOT = find_repository_root()
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
INPUT_FASTA = PROCESSED_DIR / "TP53_clean.fasta"
if not INPUT_FASTA.exists():
    raise FileNotFoundError(f"Missing input: {INPUT_FASTA}")
records = list(SeqIO.parse(INPUT_FASTA, "fasta"))
print("Sequences:", len(records))
for r in records:
    print(r.id, len(r.seq), "aa")

## 🔎 Sequence QC and Encoding

Standard amino-acid residues are converted to integer tokens; zero is reserved for padding.

In [ ]:
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
VALID = set(AMINO_ACIDS)
AA_TO_INDEX = {aa:i+1 for i, aa in enumerate(AMINO_ACIDS)}

cleaned = []
for r in records:
    seq = str(r.seq).upper().replace(" ", "").replace("\n", "")
    if set(seq) - VALID:
        print("Excluded:", r.id, sorted(set(seq)-VALID))
        continue
    cleaned.append({"id":r.id, "sequence":seq})

encoded = [[AA_TO_INDEX[aa] for aa in r["sequence"]] for r in cleaned]
MAX_LENGTH = max(map(len, encoded))
X = pad_sequences(encoded, maxlen=MAX_LENGTH, padding="post", value=0)
print("Valid sequences:", len(cleaned))
print("Encoded shape:", X.shape)

## 🧠 Sequence Representation Architecture

Protein sequence → amino-acid tokens → embedding → 1D convolution → global pooling → latent representation.

In [ ]:
EMBEDDING_DIM = 32
encoder = models.Sequential([
    layers.Input(shape=(MAX_LENGTH,)),
    layers.Embedding(len(AMINO_ACIDS)+1, EMBEDDING_DIM, mask_zero=True, name="amino_acid_embedding"),
    layers.Conv1D(64, 5, padding="same", activation="relu", name="sequence_convolution"),
    layers.GlobalMaxPooling1D(name="global_pooling"),
    layers.Dense(64, activation="relu", name="dense_representation"),
    layers.Dropout(0.20),
    layers.Dense(32, activation="relu", name="latent_representation")
], name="TP53_Sequence_Encoder")
encoder.summary()
latent = encoder(X, training=False).numpy()
print("Latent representation:", latent.shape)

In [ ]:
if len(cleaned) >= 2:
    pca = PCA(n_components=2, random_state=SEED)
    coords = pca.fit_transform(latent)
    print("Explained variance:", pca.explained_variance_ratio_)
else:
    coords = np.zeros((len(cleaned),2))

latent_df = pd.DataFrame({"id":[r["id"] for r in cleaned], "PC1":coords[:,0], "PC2":coords[:,1]})
plt.figure(figsize=(9,7))
plt.scatter(latent_df.PC1, latent_df.PC2, s=110)
for _, row in latent_df.iterrows():
    plt.annotate(row.id, (row.PC1,row.PC2), xytext=(5,5), textcoords="offset points", fontsize=8)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Exploratory TP53 Deep-Learning Latent Representation")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "tp53_deep_learning_latent_space.png", dpi=300, bbox_inches="tight")
plt.show()

## 🎯 Predictive-Modelling Architecture

The following binary classification head demonstrates how the sequence encoder could be connected to a future supervised endpoint. It is **not trained** because the current dataset lacks a sufficiently large, independently labelled cohort.

In [ ]:
inp = layers.Input(shape=(MAX_LENGTH,), name="tp53_sequence_input")
x = layers.Embedding(len(AMINO_ACIDS)+1, EMBEDDING_DIM, mask_zero=True)(inp)
x = layers.Conv1D(64, 5, padding="same", activation="relu")(x)
x = layers.GlobalMaxPooling1D()(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.30)(x)
out = layers.Dense(1, activation="sigmoid", name="predictive_output")(x)
predictive_model = models.Model(inp, out, name="TP53_Exploratory_Predictive_Model")
predictive_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
predictive_model.summary()

## ⚠️ Scientific Limitations

- The available sequence set is small for supervised deep learning.
- Sequence similarity does not establish equivalent protein function.
- TP53 conservation does not by itself demonstrate cancer resistance.
- No clinical prediction should be inferred from this notebook.
- Future validation requires larger curated datasets, independent test data, leakage control, baselines, uncertainty analysis, interpretability, and biological validation.

In [ ]:
latent_out = pd.DataFrame(latent, columns=[f"latent_feature_{i+1}" for i in range(latent.shape[1])])
latent_out.insert(0, "id", [r["id"] for r in cleaned])
latent_out.to_csv(RESULTS_DIR / "tp53_deep_learning_latent_features.csv", index=False)
latent_df.to_csv(RESULTS_DIR / "tp53_latent_pca_coordinates.csv", index=False)
print("Results exported to", RESULTS_DIR)

## 🚀 Future Research

Potential extensions include larger cross-species TP53 datasets, functional labels, evolutionary features, protein-language-model embeddings, structural descriptors, residue-level interpretability, external validation, uncertainty-aware prediction, multi-omic integration, and experimental validation.